## Video Object Tracking

This notebook runs YOLO tracking on our short MP4 videos and compares two trackers: ByteTrack and BoT-SORT.

Goal:
- Run `model.track()` on short videos in `videos/`
- Compare ByteTrack vs BoT-SORT (speed, ID behaviour)
- Save a short annotated video and a few tracked frames for the report
- Note that tracked classes come from COCO, not gas cylinders

Outputs: tracking CSVs, annotated video, tracked frame examples.

In [1]:
import time
import shutil
from pathlib import Path
from collections import Counter
import pandas as pd
import torch
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.14.1+cpu
CUDA available: False


## Setup paths

Videos in `videos/`
Outputs in `results/tracking/`

We keep only a small number of tracked frames per run.

Two trackers are compared on the same videos:
- ByteTrack — fast, stable IDs on clearly visible objects
- BoT-SORT — slightly better ID consistency, a bit slower

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

VIDEO_DIR = ROOT / "videos"
OUTPUT_DIR = ROOT / "results" / "tracking"
TRACKED_VIDEO_DIR = OUTPUT_DIR / "videos"
TRACKED_FRAME_DIR = OUTPUT_DIR / "frames"
CSV_PATH = OUTPUT_DIR / "tracking_summary.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRACKED_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
TRACKED_FRAME_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv"}
video_paths = sorted(
    p for p in VIDEO_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS
)

print(f"Video directory: {VIDEO_DIR}")
print(f"Found {len(video_paths)} videos")
for v in video_paths:
    print(f"  - {v.name}")

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Video directory: d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\videos
Found 5 videos
  - 14_55_back_left_cropped.mp4
  - 14_55_back_right_cropped.mp4
  - 14_55_front_cropped.mp4
  - Bolivia： FLEXSPEED for Side-Entry Valves (2014).mp4
  - Portugal： Petrogal LPG Cylinder Filling Plant (2013).mp4
Using device: cpu


## Load pretrained YOLO model

Same model as Notebook 1: `yolo11n.pt`, COCO-pretrained. Tracking assigns IDs on top of whatever COCO classes the model detects.

In [3]:
model = YOLO("yolo11n.pt")
coco_classes = model.names
print(f"Classes: {len(coco_classes)}")

Classes: 80


## Run both trackers on all videos

For each video and each tracker, we run `model.track()` and record:
- processing time
- average FPS
- total unique track IDs
- which classes were tracked

We save a tracked MP4 for the first video only (keeps the notebook fast), and a few tracked frames per tracker.

In [4]:
TRACKERS = ["bytetrack.yaml", "botsort.yaml"]
MAX_FRAMES = 150
SAVE_FIRST_VIDEO_ONLY = True

records = []
first_video_name = video_paths[0].name if video_paths else None

for video_path in video_paths:
    for tracker in TRACKERS:
        save_video = SAVE_FIRST_VIDEO_ONLY and (video_path.name == first_video_name)
        run_name = f"{video_path.stem}_{tracker.split('.')[0]}"

        t0 = time.time()
        results = model.track(
            source=str(video_path),
            tracker=tracker,
            conf=0.25,
            device=device,
            persist=True,
            stream=True,
            save=save_video,
            project=str(TRACKED_VIDEO_DIR),
            name=run_name,
            exist_ok=True,
            verbose=False,
        )

        frame_count = 0
        unique_ids = set()
        class_counter = Counter()

        for r in results:
            frame_count += 1
            if frame_count > MAX_FRAMES:
                break

            boxes = r.boxes
            if boxes is None or boxes.id is None:
                continue

            ids = boxes.id.cpu().numpy().astype(int)
            cls = boxes.cls.cpu().numpy().astype(int)

            unique_ids.update(ids.tolist())
            for c in cls:
                class_counter[coco_classes[int(c)]] += 1

        elapsed = time.time() - t0
        fps = frame_count / elapsed if elapsed > 0 else 0.0

        records.append({
            "video": video_path.name,
            "tracker": tracker.split(".")[0],
            "frames_processed": frame_count,
            "seconds": round(elapsed, 2),
            "fps": round(fps, 2),
            "unique_ids": len(unique_ids),
            "top_classes": dict(class_counter.most_common(3)),
            "video_saved": save_video,
        })

        print(f"{video_path.name:35s} {tracker:15s} frames={frame_count:4d} fps={fps:5.2f} ids={len(unique_ids)}")

df = pd.DataFrame(records)
df.to_csv(CSV_PATH, index=False)
print(f"\nSaved tracking summary to {CSV_PATH.relative_to(ROOT)}")

14_55_back_left_cropped.mp4         bytetrack.yaml  frames= 151 fps= 6.53 ids=4
14_55_back_left_cropped.mp4         botsort.yaml    frames= 151 fps= 6.11 ids=4
14_55_back_right_cropped.mp4        bytetrack.yaml  frames= 151 fps=18.48 ids=9
14_55_back_right_cropped.mp4        botsort.yaml    frames= 151 fps=10.69 ids=10
14_55_front_cropped.mp4             bytetrack.yaml  frames= 151 fps=16.58 ids=4
14_55_front_cropped.mp4             botsort.yaml    frames= 151 fps=10.95 ids=4
Bolivia： FLEXSPEED for Side-Entry Valves (2014).mp4 bytetrack.yaml  frames= 151 fps=23.13 ids=0
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WAR

In [5]:
import cv2

FRAMES_PER_VIDEO = 10

for video_path in video_paths:
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if total_frames <= 0:
        print(f"[skip] {video_path.name}: cannot read frame count")
        continue

    step = max(1, total_frames // FRAMES_PER_VIDEO)
    sample_frames = list(range(0, total_frames, step))[:FRAMES_PER_VIDEO]

    for tracker in TRACKERS:
        tracker_name = tracker.split(".")[0]
        out_dir = TRACKED_FRAME_DIR / video_path.stem / tracker_name
        out_dir.mkdir(parents=True, exist_ok=True)

        cap = cv2.VideoCapture(str(video_path))
        idx = 0
        saved = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if idx in sample_frames:
                res = model.track(
                    source=frame,
                    tracker=tracker,
                    conf=0.25,
                    device=device,
                    persist=True,
                    verbose=False,
                )[0]

                annotated = res.plot()
                out_path = out_dir / f"frame_{idx:04d}.jpg"
                cv2.imwrite(str(out_path), annotated)
                saved += 1

            idx += 1

        cap.release()
        print(f"{video_path.name:35s} {tracker_name:15s} saved={saved}")

14_55_back_left_cropped.mp4         bytetrack       saved=10
14_55_back_left_cropped.mp4         botsort         saved=10
14_55_back_right_cropped.mp4        bytetrack       saved=10
14_55_back_right_cropped.mp4        botsort         saved=10
14_55_front_cropped.mp4             bytetrack       saved=10
14_55_front_cropped.mp4             botsort         saved=10
Bolivia： FLEXSPEED for Side-Entry Valves (2014).mp4 bytetrack       saved=10
WARNING not enough matching points
Bolivia： FLEXSPEED for Side-Entry Valves (2014).mp4 botsort         saved=10
Portugal： Petrogal LPG Cylinder Filling Plant (2013).mp4 bytetrack       saved=10
Portugal： Petrogal LPG Cylinder Filling Plant (2013).mp4 botsort         saved=10


## Summary

Aggregate tracking results for the report: per-tracker average FPS, average unique IDs, and which classes were most tracked.

In [6]:
print("=" * 60)
print("TRACKING SUMMARY")
print("=" * 60)

for tracker_name in ["bytetrack", "botsort"]:
    sub = df[df["tracker"] == tracker_name]
    if sub.empty:
        continue
    print(f"\n{tracker_name.upper()}")
    print(f"  videos tested:       {len(sub)}")
    print(f"  avg FPS:             {sub['fps'].mean():.2f}")
    print(f"  avg unique IDs:      {sub['unique_ids'].mean():.1f}")
    print(f"  total frames:        {int(sub['frames_processed'].sum())}")

print("\nPer-video breakdown:")
print(df.to_string(index=False))

print(f"\nFrames saved under: {TRACKED_FRAME_DIR.relative_to(ROOT)}")

TRACKING SUMMARY

BYTETRACK
  videos tested:       5
  avg FPS:             17.26
  avg unique IDs:      4.2
  total frames:        755

BOTSORT
  videos tested:       5
  avg FPS:             12.41
  avg unique IDs:      4.6
  total frames:        755

Per-video breakdown:
                                                   video   tracker  frames_processed  seconds   fps  unique_ids                                   top_classes  video_saved
                             14_55_back_left_cropped.mp4 bytetrack               151    23.13  6.53           4          {'vase': 40, 'person': 8, 'cake': 3}         True
                             14_55_back_left_cropped.mp4   botsort               151    24.73  6.11           4          {'vase': 40, 'person': 8, 'cake': 3}         True
                            14_55_back_right_cropped.mp4 bytetrack               151     8.17 18.48           9       {'chair': 107, 'vase': 45, 'spoon': 25}        False
                            14_55_back_ri

## Tracking with fine-tuned models

Here we reuse the fine-tuned models from Notebook 3 and the CVAT notebook, and re-run tracking on the same videos.

The purpose is to see whether the fine-tuned detectors produce different tracks — specifically, whether they detect cylinders or cap regions that the COCO model did not.

We use the same trackers (ByteTrack, BoT-SORT) and save 5 tracked frames per video per model.

In [7]:
FT_MODELS = {
    "gas_bottles": ROOT / "results" / "gas_bottle_training" / "yolo11n_gas_bottles" / "weights" / "best.pt",
    "gas_cylinder": ROOT / "results" / "gas_bottle_training" / "yolo11n_gas_cylinder" / "weights" / "best.pt",
    "cvat_batch_1": ROOT / "results" / "cvat_batch_1" / "training" / "yolo11n_cvat_batch_1" / "weights" / "best.pt",
}

available_ft_models = {}
for name, path in FT_MODELS.items():
    exists = path.exists()
    print(f"{name:15s} {path.relative_to(ROOT)}  exists={exists}")
    if exists:
        available_ft_models[name] = path

print(f"\nAvailable fine-tuned models: {list(available_ft_models.keys())}")

gas_bottles     results\gas_bottle_training\yolo11n_gas_bottles\weights\best.pt  exists=True
gas_cylinder    results\gas_bottle_training\yolo11n_gas_cylinder\weights\best.pt  exists=True
cvat_batch_1    results\cvat_batch_1\training\yolo11n_cvat_batch_1\weights\best.pt  exists=True

Available fine-tuned models: ['gas_bottles', 'gas_cylinder', 'cvat_batch_1']


## Run tracking with each fine-tuned model

For every video and every available fine-tuned model we run ByteTrack and record FPS, unique IDs, and which classes were detected.

In [8]:
FT_TRACKER = "bytetrack.yaml"
FT_MAX_FRAMES = 150

ft_records = []

for model_name, model_path in available_ft_models.items():
    ft_model = YOLO(str(model_path))
    ft_classes = ft_model.names

    for video_path in video_paths:
        run_name = f"ft_{model_name}_{video_path.stem}_{FT_TRACKER.split('.')[0]}"
        save_video = (video_path.name == first_video_name)

        t0 = time.time()
        results = ft_model.track(
            source=str(video_path),
            tracker=FT_TRACKER,
            conf=0.25,
            device=device,
            persist=True,
            stream=True,
            save=save_video,
            project=str(TRACKED_VIDEO_DIR),
            name=run_name,
            exist_ok=True,
            verbose=False,
        )

        frame_count = 0
        unique_ids = set()
        class_counter = Counter()

        for r in results:
            frame_count += 1
            if frame_count > FT_MAX_FRAMES:
                break

            boxes = r.boxes
            if boxes is None or boxes.id is None:
                continue

            ids = boxes.id.cpu().numpy().astype(int)
            cls = boxes.cls.cpu().numpy().astype(int)

            unique_ids.update(ids.tolist())
            for c in cls:
                class_counter[ft_classes[int(c)]] += 1

        elapsed = time.time() - t0
        fps = frame_count / elapsed if elapsed > 0 else 0.0

        ft_records.append({
            "model": model_name,
            "video": video_path.name,
            "tracker": FT_TRACKER.split(".")[0],
            "frames_processed": frame_count,
            "seconds": round(elapsed, 2),
            "fps": round(fps, 2),
            "unique_ids": len(unique_ids),
            "top_classes": dict(class_counter.most_common(3)),
            "video_saved": save_video,
        })

        print(f"{model_name:15s} {video_path.name:35s} frames={frame_count:4d} fps={fps:5.2f} ids={len(unique_ids)}")

df_ft = pd.DataFrame(ft_records)
FT_CSV_PATH = OUTPUT_DIR / "tracking_finetuned_summary.csv"
df_ft.to_csv(FT_CSV_PATH, index=False)
print(f"\nSaved fine-tuned tracking summary to {FT_CSV_PATH.relative_to(ROOT)}")

gas_bottles     14_55_back_left_cropped.mp4         frames= 151 fps= 9.70 ids=2
gas_bottles     14_55_back_right_cropped.mp4        frames= 151 fps=23.10 ids=2
gas_bottles     14_55_front_cropped.mp4             frames= 151 fps=22.88 ids=4
gas_bottles     Bolivia： FLEXSPEED for Side-Entry Valves (2014).mp4 frames= 151 fps=31.12 ids=2
gas_bottles     Portugal： Petrogal LPG Cylinder Filling Plant (2013).mp4 frames= 151 fps=33.84 ids=1
gas_cylinder    14_55_back_left_cropped.mp4         frames= 151 fps= 9.85 ids=0
gas_cylinder    14_55_back_right_cropped.mp4        frames= 151 fps=24.05 ids=0
gas_cylinder    14_55_front_cropped.mp4             frames= 151 fps=24.15 ids=0
gas_cylinder    Bolivia： FLEXSPEED for Side-Entry Valves (2014).mp4 frames= 151 fps=28.33 ids=0
gas_cylinder    Portugal： Petrogal LPG Cylinder Filling Plant (2013).mp4 frames= 151 fps=35.70 ids=0
cvat_batch_1    14_55_back_left_cropped.mp4         frames= 151 fps= 8.46 ids=3
cvat_batch_1    14_55_back_right_cropped.mp4  

## Saved frames for fine-tuned model

Frames are saved under `results/tracking/frames_ft/<model>/<video>/`.

In [9]:
FT_FRAMES_PER_VIDEO = 5
FT_FRAME_DIR = OUTPUT_DIR / "frames_ft"

for model_name, model_path in available_ft_models.items():
    ft_model = YOLO(str(model_path))

    for video_path in video_paths:
        cap = cv2.VideoCapture(str(video_path))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()

        if total_frames <= 0:
            print(f"[skip] {video_path.name}: cannot read frame count")
            continue

        step = max(1, total_frames // FT_FRAMES_PER_VIDEO)
        sample_frames = list(range(0, total_frames, step))[:FT_FRAMES_PER_VIDEO]

        out_dir = FT_FRAME_DIR / model_name / video_path.stem
        out_dir.mkdir(parents=True, exist_ok=True)

        cap = cv2.VideoCapture(str(video_path))
        idx = 0
        saved = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if idx in sample_frames:
                res = ft_model.track(
                    source=frame,
                    tracker=FT_TRACKER,
                    conf=0.25,
                    device=device,
                    persist=True,
                    verbose=False,
                )[0]

                annotated = res.plot()
                out_path = out_dir / f"frame_{idx:04d}.jpg"
                cv2.imwrite(str(out_path), annotated)
                saved += 1

            idx += 1

        cap.release()
        print(f"{model_name:15s} {video_path.name:35s} saved={saved}")

print(f"\nFine-tuned tracked frames saved under {FT_FRAME_DIR.relative_to(ROOT)}")

gas_bottles     14_55_back_left_cropped.mp4         saved=5
gas_bottles     14_55_back_right_cropped.mp4        saved=5
gas_bottles     14_55_front_cropped.mp4             saved=5
gas_bottles     Bolivia： FLEXSPEED for Side-Entry Valves (2014).mp4 saved=5
gas_bottles     Portugal： Petrogal LPG Cylinder Filling Plant (2013).mp4 saved=5
gas_cylinder    14_55_back_left_cropped.mp4         saved=5
gas_cylinder    14_55_back_right_cropped.mp4        saved=5
gas_cylinder    14_55_front_cropped.mp4             saved=5
gas_cylinder    Bolivia： FLEXSPEED for Side-Entry Valves (2014).mp4 saved=5
gas_cylinder    Portugal： Petrogal LPG Cylinder Filling Plant (2013).mp4 saved=5
cvat_batch_1    14_55_back_left_cropped.mp4         saved=5
cvat_batch_1    14_55_back_right_cropped.mp4        saved=5
cvat_batch_1    14_55_front_cropped.mp4             saved=5
cvat_batch_1    Bolivia： FLEXSPEED for Side-Entry Valves (2014).mp4 saved=5
cvat_batch_1    Portugal： Petrogal LPG Cylinder Filling Plant (2013).m

## Summary — fine-tuned tracking

Aggregate FPS, unique IDs, and classes for each fine-tuned model across all videos.

In [10]:
print("=" * 60)
print("FINE-TUNED TRACKING SUMMARY")
print("=" * 60)

for model_name in df_ft["model"].unique():
    sub = df_ft[df_ft["model"] == model_name]
    print(f"\n{model_name}")
    print(f"  videos tested:   {len(sub)}")
    print(f"  avg FPS:         {sub['fps'].mean():.2f}")
    print(f"  avg unique IDs:  {sub['unique_ids'].mean():.1f}")
    print(f"  total frames:    {int(sub['frames_processed'].sum())}")

print("\nPer-video breakdown:")
print(df_ft.to_string(index=False))

print(f"\nSummary CSV:   {FT_CSV_PATH.relative_to(ROOT)}")
print(f"Frames folder: {FT_FRAME_DIR.relative_to(ROOT)}")

FINE-TUNED TRACKING SUMMARY

gas_bottles
  videos tested:   5
  avg FPS:         24.13
  avg unique IDs:  2.2
  total frames:    755

gas_cylinder
  videos tested:   5
  avg FPS:         24.42
  avg unique IDs:  0.0
  total frames:    755

cvat_batch_1
  videos tested:   5
  avg FPS:         19.46
  avg unique IDs:  3.2
  total frames:    755

Per-video breakdown:
       model                                                    video   tracker  frames_processed  seconds   fps  unique_ids           top_classes  video_saved
 gas_bottles                              14_55_back_left_cropped.mp4 bytetrack               151    15.57  9.70           2 {'gas_cylinder': 153}         True
 gas_bottles                             14_55_back_right_cropped.mp4 bytetrack               151     6.54 23.10           2 {'gas_cylinder': 157}        False
 gas_bottles                                  14_55_front_cropped.mp4 bytetrack               151     6.60 22.88           4 {'gas_cylinder': 164}       